# Image Metadata Viewer

This notebook loads images from a folder and displays them with their metadata (title and tags) from the MetadataManager.

## 1. Import Required Libraries

Import necessary libraries including os, pathlib, PIL for image handling, and the MetadataManager from phototag.

In [ ]:
import os
import re

# import sys
import textwrap
from datetime import datetime, timedelta
from pathlib import Path
from typing import Protocol

import matplotlib.pyplot as plt
from dotenv import load_dotenv
from matplotlib.gridspec import GridSpec
from PIL import Image

from phototag.db import Db
from phototag.metadata_manager import MetadataManager
from phototag.phototag import PhotoTag

# Add the src directory to the path
# sys.path.insert(0, str(Path.cwd() / "src"))

## 1. Define display and file filter functions

Implement the function to display the metadata for images

In [ ]:
class ImageFilter(Protocol):
    """Protocol for image filtering functions."""

    def __call__(self, images: list[Path]) -> list[Path]:
        """Filter a set of image paths and return the filtered set."""
        ...


def create_age_filter(max_days: int) -> ImageFilter:
    """Create a filter function that keeps only images modified within the last N days.

    Args:
        max_days: Maximum age of files in days

    Returns:
        A filter function that takes a set of image paths and returns filtered paths
    """

    def filter_by_age(images: list[Path]) -> list[Path]:
        cutoff = datetime.now() - timedelta(days=max_days)
        return [
            image_path
            for image_path in images
            if datetime.fromtimestamp(image_path.stat().st_mtime) >= cutoff
        ]

    return filter_by_age


def create_max_files_filter(max_files: int) -> ImageFilter:
    """Create a filter function that limits to the first N files.

    Args:
        max_files: Maximum number of files to keep

    Returns:
        A filter function that takes a set of image paths and returns filtered paths
    """

    def filter_by_max_files(images: list[Path]) -> list[Path]:
        return images[:max_files]

    return filter_by_max_files


def create_regexp_filter(pattern: str) -> ImageFilter:
    """Create a filter function that matches filenames against a regexp pattern.

    Args:
        pattern: Regular expression pattern to match filenames

    Returns:
        A filter function that takes a set of image paths and returns filtered paths
    """
    regex = re.compile(pattern)

    def filter_by_regexp(images: list[Path]) -> list[Path]:
        return [image_path for image_path in images if regex.search(image_path.name)]

    return filter_by_regexp


def create_db_filter(metadata_managaer: MetadataManager) -> ImageFilter:
    """Create a filter function that keeps only images with existing database entries.

    Args:
        metadata_managaer: Metadata manager instance to check existing filenames

    Returns:
        A filter function that takes a set of image paths and returns filtered paths
    """

    def filter_by_db(images: list[Path]) -> list[Path]:
        existing = {record.filename for record in metadata_managaer.all()}
        return [
            image_path
            for image_path in images
            if str(image_path) in existing or image_path.name in existing
        ]

    return filter_by_db


def load_images(source_folder: Path, extensions: set[str]) -> list[Path]:
    """Load image paths from the source folder with specified extensions.

    Args:
        source_folder: Path to the folder containing images
        extensions: Set of file extensions to include (e.g. {'.jpg', '.png'})
    """
    image_files: list[Path] = []
    for ext in extensions:
        image_files.extend(source_folder.glob(f"*{ext}"))
        image_files.extend(source_folder.glob(f"*{ext.upper()}"))

    image_files = sorted(set(image_files))  # Remove duplicates and sort
    return image_files


def display_image_with_metadata(image_path: Path, meta_manager: MetadataManager):
    """Display a single image with its metadata."""
    try:
        # Get metadata from manager
        metadata = meta_manager.get_by_filename(str(image_path))

        # Load and display image
        img = Image.open(image_path)

        # Create figure with image and metadata
        fig = plt.figure(figsize=(12, 6))
        gs = GridSpec(1, 2, width_ratios=[2, 1], figure=fig)

        # Display image
        ax_img = fig.add_subplot(gs[0])
        ax_img.imshow(img)
        ax_img.set_title(image_path.name, fontsize=12, fontweight="bold")
        ax_img.axis("off")

        # Display metadata
        ax_meta = fig.add_subplot(gs[1])
        ax_meta.axis("off")

        meta_text = f"File: {image_path.name}\n\n"
        raw_text = f"{image_path.name}\n"

        if metadata:
            # Wrap long title
            title = metadata.title or "N/A"
            wrapped_title = textwrap.fill(title, width=40)
            meta_text += f"Title: {wrapped_title}\n\n"
            raw_text += f"{title}\n"

            # Wrap and format tags
            if metadata.keywords:
                tags_text = ", ".join(metadata.keywords)
                wrapped_tags = textwrap.fill(f"Tags: {tags_text}", width=40)
                meta_text += f"{wrapped_tags}\n\n"
                raw_text += f"{tags_text}\n"
            else:
                meta_text += "Tags: No tags\n\n"
                raw_text += f"No tags\n"

            # Wrap long description
            description = metadata.description or "N/A"
            wrapped_desc = textwrap.fill(f"Description: {description}", width=40)
            meta_text += wrapped_desc
            raw_text += f"{description}\n"
        else:
            meta_text += "No metadata found in database"

        ax_meta.text(
            0.05,
            0.95,
            meta_text,
            transform=ax_meta.transAxes,
            fontsize=9,
            verticalalignment="top",
            fontfamily="monospace",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3),
        )

        plt.tight_layout()
        plt.show()
        print(raw_text)

    except Exception as e:
        print(f"Error displaying {image_path.name}: {e}")

## 2. Define Source Folder Parameter

Set the source folder path as a parameter that can be easily modified. Point to the directory containing images to process.

In [ ]:
# Load environment variables from ~/.phototag.env
load_dotenv(dotenv_path=Path.home() / ".phototag.env")

# Get configuration from environment variables
URL = os.getenv("PHOTOTAG_URL", "https://server.phototag.ai/api/keywords")
TOKEN = os.getenv("PHOTOTAG_TOKEN", "")
DB_FILE = os.getenv("PHOTOTAG_DB", str(Path.home() / ".phototag_db.json"))

# Validate that token is available
if not TOKEN:
    print(
        "Warning: PHOTOTAG_TOKEN not found in environment. Set it in ~/.phototag.env or as an environment variable."
    )

# Initialize Db and PhotoTag
db = Db(DB_FILE)
phototag = PhotoTag(
    url=URL,
    token=TOKEN,
)

# Create MetadataManager instance
meta_manager = MetadataManager(db, phototag)

print("MetadataManager initialized")
print(f"API URL: {URL}")
print(f"Database file: {DB_FILE}")
with db as c:
    print(f"Database record count: {c.len()}")

## 4. Load and Display Images with Metadata

Iterate through all image files in the source folder, retrieve existing metadata (title and tags) from MetadataManager for each image, and display the image with its associated metadata.

In [ ]:
# Configuration - modify this to change the source folder
from venv import create

SOURCE_FOLDER = (
    Path.home() / "Pictures" / "Lightroom Saved Photos"
)  # Change this to your image folder
MAX_FILE_AGE_DAYS = -1  # Change this value to keep only files newer than N days, or set to None to disable
MAX_FILES = -1  # Change this value to limit the number of displayed images, or set to None to disable
REGEXP = "expo*"  # Change this to a regular expression to filter filenames, or set to None to disable
ONLY_EXISTING_IN_DB = True  # Set to True to keep only images that have entries in the database, or False to disable

# Supported image extensions
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".webp", ".tiff"}

# Verify the source folder exists
if not SOURCE_FOLDER.exists():
    print(f"Error: Source folder does not exist: {SOURCE_FOLDER}")
    raise FileNotFoundError(f"Source folder does not exist: {SOURCE_FOLDER}")
else:
    print(f"Source folder: {SOURCE_FOLDER}")

# Get list of image files
image_files = load_images(SOURCE_FOLDER, list(SUPPORTED_EXTENSIONS))

print(f"Found {len(image_files)} image files")

filter: ImageFilter

# Keep only files modified within the last N days
if MAX_FILE_AGE_DAYS is not None and MAX_FILE_AGE_DAYS > 0:
    filter = create_age_filter(MAX_FILE_AGE_DAYS)
    image_files = filter(image_files)
    print(
        f"Filtered to {len(image_files)} images modified in the last {MAX_FILE_AGE_DAYS} days"
    )

if ONLY_EXISTING_IN_DB:
    filter = create_db_filter(meta_manager)
    image_files = filter(image_files)
    print(f"Filtered to {len(image_files)} images that have entries in the database")

if REGEXP is not None:
    filter = create_regexp_filter(REGEXP)
    image_files = filter(image_files)
    print(f"Filtered to {len(image_files)} images matching regexp: {REGEXP}")

if MAX_FILES is not None and MAX_FILES > 0:
    filter = create_max_files_filter(MAX_FILES)
    image_files = filter(image_files)
    print(f"Filtered to {len(image_files)} images limited to max {MAX_FILES} files")


for image_path in image_files:
    display_image_with_metadata(image_path, meta_manager)